# Pipeline de datos

De los CSV originales a las tablas de modelado `data/interim/{train,val,test}.parquet`. Cada fila es una máquina-hora, con las 31 variables de 1.4 y las etiquetas de 1.3. Este notebook debe ejecutarse **antes** que `04_features.ipynb` y `05_baseline.ipynb`, que leen esas tablas.

Pasos:

1. Cargar los cinco CSV.
2. Calcular las variables (`src/features.py`): sensores, errores, mantenimiento y estáticas.
3. Calcular las etiquetas (`src/labels.py`): `y_bin`, `y_cls` y `h_to_failure`.
4. Asignar cada hora a un conjunto, dejando un margen de 24 h en cada frontera.
5. Guardar una tabla por conjunto. `test.parquet` se escribe aparte y no se abre hasta la semana 3.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import pandas as pd

from src.data import load_all, INTERIM_DIR
from src.features import (ALL_FEATURES, error_features, maint_features, sensor_features,
                          static_features)
from src.labels import assign_label_split, make_labels

KEYS = ["machineID", "datetime"]
LABEL_COLS = ["y_bin", "y_cls", "h_to_failure"]

## 1. Cargar los CSV

In [2]:
d = load_all()
{name: df.shape for name, df in d.items()}

{'telemetry': (876100, 6),
 'errors': (3919, 3),
 'maint': (3286, 3),
 'failures': (761, 3),
 'machines': (100, 3)}

## 2. Variables

Todas son causales: la fila (*m*, *t*) usa solo información con marca de tiempo ≤ *t*. La rejilla de horas es la de la telemetría.

In [3]:
sens = sensor_features(d["telemetry"])
grid = sens[KEYS]

feats = sens
for part in (error_features(grid, d["errors"]),
             maint_features(grid, d["maint"]),
             static_features(grid, d["machines"])):
    feats = feats.merge(part, on=KEYS, how="left", validate="one_to_one")
feats = feats[KEYS + ALL_FEATURES]
print(feats.shape)
feats.head()

(876100, 33)


,machineID,datetime,volt_mean_3h,volt_std_3h,rotate_mean_3h,rotate_std_3h,pressure_mean_3h,pressure_std_3h,vibration_mean_3h,vibration_std_3h,...,error_total_count_24h,h_since_comp1,h_since_comp2,h_since_comp3,h_since_comp4,model_model1,model_model2,model_model3,model_model4,age
0,1,2015-01-01 06:00:00,176.217853,NaN,418.504078,NaN,113.077935,NaN,45.087686,NaN,...,0,456.0,5136.0,3696.0,4056.0,0,0,1,0,18
1,1,2015-01-01 07:00:00,169.548538,9.431836,410.625784,11.141591,104.269230,12.457390,44.250829,1.183494,...,0,457.0,5137.0,3697.0,4057.0,0,0,1,0,18
2,1,2015-01-01 08:00:00,170.028993,6.721032,449.533798,67.849599,94.592122,18.934956,40.893502,5.874970,...,0,458.0,5138.0,3698.0,4058.0,0,0,1,0,18
3,1,2015-01-01 09:00:00,165.443986,4.807415,425.415550,92.702671,93.315664,17.106476,39.571655,4.808836,...,0,459.0,5139.0,3699.0,4059.0,0,0,1,0,18
4,1,2015-01-01 10:00:00,163.687586,6.773501,436.292011,90.603712,98.791038,20.440216,33.763834,7.574349,...,0,460.0,5140.0,3700.0,4060.0,0,0,1,0,18


## 3. Etiquetas

In [4]:
labels = make_labels(grid, d["failures"])
labels[LABEL_COLS].describe()

,y_bin,y_cls,h_to_failure
count,876100.000000,876100.000000,755256.000000
mean,0.019614,0.047419,912.039749
std,0.138670,0.369583,952.124898
min,0.000000,0.000000,1.000000
25%,0.000000,0.000000,274.000000
50%,0.000000,0.000000,627.000000
75%,0.000000,0.000000,1211.000000
max,1.000000,4.000000,6480.000000


## 4. Unir y asignar conjunto

In [5]:
df = feats.merge(labels, on=KEYS, how="left", validate="one_to_one")
df["split"] = assign_label_split(df["datetime"]).to_numpy()
df.groupby("split", dropna=False).agg(horas=("y_bin", "size"), positivos=("y_bin", "mean"),
                                      desde=("datetime", "min"), hasta=("datetime", "max"))

,horas,positivos,desde,hasta
split,,,,
test,144000,0.018458,2015-11-01 00:00:00,2015-12-30 23:00:00
train,580200,0.020238,2015-01-01 06:00:00,2015-08-30 23:00:00
val,144000,0.018417,2015-09-01 00:00:00,2015-10-30 23:00:00
NaN,7900,0.016709,2015-08-31 00:00:00,2016-01-01 06:00:00


Las horas con `split` vacío son las del margen entre conjuntos y se descartan.

## 5. Guardar

In [6]:
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
for s in ("train", "val", "test"):
    out = df.loc[df["split"] == s, KEYS + ALL_FEATURES + LABEL_COLS]
    out.to_parquet(INTERIM_DIR / f"{s}.parquet", index=False)
    print(f"{s:5s} {out.shape}  → {(INTERIM_DIR / f'{s}.parquet').relative_to(ROOT)}")

train (580200, 36)  → data/interim/train.parquet
val   (144000, 36)  → data/interim/val.parquet


test  (144000, 36)  → data/interim/test.parquet
